# Phase 3a (prep) — Embed the FiQA-2018 scaling corpus with E5-base-v2

Phase 3's headline question — *at what corpus size does an ANN index (HNSW/IVF) actually beat a brute-force matmul?* — needs a **real** corpus an order of magnitude larger than SciFact (5k) / NFCorpus (3.6k). FiQA-2018 (BeIR, financial-domain QA, ~57k passages) is that corpus.

This notebook does **only the torch work**: load FiQA, embed docs + queries with E5-base-v2, persist `.npy` + id/qrels JSON. It imports **no faiss** — the main Phase-3 benchmark notebook (`phase3_faiss_index.ipynb`) is faiss-only. Keeping the two libomp users (torch, faiss) in separate kernels is the documented fix for the Apple-Silicon faiss/libomp deadlock noted since Phase 1.

In [1]:
import os, json, time, gc, warnings
from collections import defaultdict
os.environ["TOKENIZERS_PARALLELISM"] = "false"; os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["OMP_NUM_THREADS"] = "4"
import numpy as np
warnings.filterwarnings("ignore")
SEED = 42; np.random.seed(SEED)
import torch; torch.set_num_threads(4)
DEVICE = "cpu"
EMB_DIR = "../data/processed/emb_cache"; os.makedirs(EMB_DIR, exist_ok=True)
META_DIR = "../data/processed/meta"; os.makedirs(META_DIR, exist_ok=True)
print("torch", torch.__version__, "| device", DEVICE)

torch 2.12.0 | device cpu


In [2]:
# Load FiQA-2018 exactly the way Phase 1/2 loaded SciFact/NFCorpus (same dict-insertion order
# => the cached .npy rows line up 1:1 with list(docs) in the faiss notebook).
from datasets import load_dataset
import datasets as _ds; _ds.disable_progress_bars()
def load_beir(name, qrels_split="test"):
    corpus  = load_dataset(f"BeIR/{name}", "corpus",  split="corpus")
    queries = load_dataset(f"BeIR/{name}", "queries", split="queries")
    qrels_t = load_dataset(f"BeIR/{name}-qrels", split=qrels_split)
    docs = {str(r["_id"]): (r["title"]+". "+r["text"]).strip() if r["title"] else r["text"] for r in corpus}
    qtext = {str(r["_id"]): r["text"] for r in queries}
    qrels = defaultdict(dict)
    for r in qrels_t: qrels[str(r["query-id"])][str(r["corpus-id"])] = int(r["score"])
    qtext = {q: t for q, t in qtext.items() if q in qrels}
    return docs, qtext, dict(qrels)
t0=time.time()
docs, queries, qrels = load_beir("fiqa")
print(f"fiqa: {len(docs)} docs | {len(queries)} queries | "
      f"{np.mean([len(v) for v in qrels.values()]):.1f} rel/query | load {time.time()-t0:.1f}s")
print("doc-length (chars) pct:", np.percentile([len(v) for v in docs.values()],[50,90,99]).round(0))

fiqa: 57638 docs | 648 queries | 2.6 rel/query | load 17.3s
doc-length (chars) pct: [ 522. 1551. 3698.]


In [3]:
# Embed with E5-base-v2 (same prefixes/normalisation as Phase 2's MODELS['E5-base-v2']).
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer("intfloat/e5-base-v2", device=DEVICE); enc.max_seq_length = 512
def embed(texts, prefix, tag, bs=64):
    fp = f"{EMB_DIR}/E5-base-v2__{tag}.npy"
    if os.path.exists(fp):
        v = np.load(fp)
        if len(v) == len(texts): print(f"cache hit {tag}: {v.shape}"); return v
    t0=time.time()
    v = enc.encode([prefix+t for t in texts], batch_size=bs, convert_to_numpy=True,
                   normalize_embeddings=True, show_progress_bar=False).astype("float32")
    np.save(fp, v); print(f"embedded {tag}: {v.shape} in {time.time()-t0:.0f}s")
    return v
doc_ids = list(docs); q_ids = list(queries)
D = embed([docs[d] for d in doc_ids], "passage: ", "fiqa_docs")
Q = embed([queries[q] for q in q_ids], "query: ", "fiqa_q")
print("D", D.shape, "Q", Q.shape)

embedded fiqa_docs: (57638, 768) in 2323s


embedded fiqa_q: (648, 768) in 2s
D (57638, 768) Q (648, 768)


In [4]:
# Persist id-order + qrels so the faiss notebook never needs torch/datasets to align rows.
json.dump(doc_ids, open(f"{META_DIR}/fiqa_doc_ids.json", "w"))
json.dump(q_ids,   open(f"{META_DIR}/fiqa_q_ids.json",  "w"))
json.dump(qrels,   open(f"{META_DIR}/fiqa_qrels.json",  "w"))
# sanity: rows align, norms ~1
assert D.shape[0] == len(doc_ids) and Q.shape[0] == len(q_ids)
print("norms doc/q:", round(float(np.linalg.norm(D[0])),3), round(float(np.linalg.norm(Q[0])),3))
print("saved meta to", META_DIR)
print("DONE phase3a")

norms doc/q: 1.0 1.0
saved meta to ../data/processed/meta
DONE phase3a
